# Ill-conditioned overlap handling with `TBSystem`

This notebook demonstrates the unified `TBSystem` API for band post-processing with an explicit `ill_threshold` fallback. The fallback is intended for inference/post-processing when the overlap matrix `S(k)` is ill-conditioned. It should not be enabled silently during training.

For non-orthogonal models DeePTB solves

```text
H(k) c = E S(k) c
```

If `S(k)` has very small eigenvalues, Cholesky factorization may fail. Setting `ill_threshold` projects out overlap modes below that threshold and solves the generalized eigenvalue problem in the remaining well-conditioned subspace.

In [ ]:
from pathlib import Path

import numpy as np

from dptb.data import AtomicDataDict
from dptb.postprocess.unified.system import TBSystem

In [ ]:
# This works whether the notebook is launched from the repo root or this example directory.
example_dir = Path.cwd()
if not (example_dir / "band_with_ill_overlap.json").exists():
    example_dir = Path("examples/ill_conditioned_overlap").resolve()
root = example_dir.parent

model_path = root / "e3/ref_model/nnenv.ep1474.pth"
structure_path = root / "e3/data/Si64.vasp"
overlap_path = root / "e3/data/Si64.0/overlaps.h5"

model_path, structure_path, overlap_path

In [ ]:
system = TBSystem(
    data=str(structure_path),
    calculator=str(model_path),
    override_overlap=str(overlap_path),
    device="cpu",
)

In [ ]:
kpath_config = {
    "method": "abacus",
    "kpath": [
        [0.0, 0.0, 0.0, 20],
        [0.5, 0.0, 0.0, 1],
    ],
    "klabels": ["G", "X"],
}

bands = system.get_bands(
    kpath_config=kpath_config,
    ill_threshold=5e-4,
    ill_pad_value=10000.0,
    reuse=False,
)

bands.band_data.eigenvalues.shape

In [ ]:
eigs = bands.band_data.eigenvalues
print("eigenvalue shape:", eigs.shape)
print("energy range:", np.nanmin(eigs), np.nanmax(eigs))

When overlap modes are projected out, DeePTB stores a validity mask. Entries marked `False` correspond to padding values, not physical eigenvalues. In this bundled silicon example the supplied overlap may be well-conditioned, so the mask can be absent.

In [ ]:
mask = system.data.get(AtomicDataDict.EIGENVALUE_VALID_MASK_KEY)
if mask is None:
    print("No overlap modes were projected out for this run.")
else:
    mask = mask[0].detach().cpu().numpy()
    print("valid eigenvalue mask shape:", mask.shape)
    print("projected-out entries:", mask.size - mask.sum())

Plotting works as usual. The plot uses padded eigenvalues only to preserve array shape; downstream code should consult the validity mask if projected modes are present.

In [ ]:
bands.plot(filename="ill_overlap_band.png", emin=-7, emax=18, show=False)